## Imports

In [1]:

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression


## Dataset import

We must use the unprocessed data, because the preprocessing done before is not useful for this regression task


In [2]:
import pandas as pd

# Percorsi corretti ai CSV (usa stringhe raw per evitare errori di escape)
train_path = r"C:\Users\loren\OneDrive\Desktop\PoliTo\machine Learning\https_training.csv"
test_path  = r"C:\Users\loren\OneDrive\Desktop\PoliTo\machine Learning\https_test.csv"

# Carica e mantieni solo le colonne numeriche (prefisso '_')
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

numeric_cols = [c for c in df_train.columns if c.startswith('_')]
df_train = df_train[numeric_cols].copy()
df_test = df_test[numeric_cols].copy()



In [3]:

import numpy as np

def regression_metrics(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "R2": r2_score(y_true, y_pred)
    }


Data preparation (s_bytes_all)

In [4]:

# Target: s_bytes_all
ytr_bytes = df_train['_s_bytes_all'].copy()
yte_bytes = df_test['_s_bytes_all'].copy()

# Remove forbidden column
Xtr_bytes = df_train.drop(columns=['_s_bytes_all', '_s_bytes_uniq'], errors='ignore')
Xte_bytes = df_test.drop(columns=['_s_bytes_all', '_s_bytes_uniq'], errors='ignore')


Data preparation (s_rtt_avg)

In [5]:

# Remove zero RTT values
train_rtt = df_train[df_train['_s_rtt_avg'] != 0].copy()
test_rtt = df_test[df_test['_s_rtt_avg'] != 0].copy()

ytr_rtt = train_rtt['_s_rtt_avg']
yte_rtt = test_rtt['_s_rtt_avg']

# Remove all RTT-related columns from features
rtt_cols = [c for c in train_rtt.columns if 'rtt' in c.lower()]

Xtr_rtt = train_rtt.drop(columns=rtt_cols, errors='ignore')
Xte_rtt = test_rtt.drop(columns=rtt_cols, errors='ignore')


Cell 5 — Define models

In [6]:

models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'HistGradientBoostingRegressor': HistGradientBoostingRegressor(random_state=42)
}


Cell 6 — Train & evaluate (default hyperparameters)

In [7]:

tasks = {
    'bytes': {
        'X_train': Xtr_bytes,
        'y_train': ytr_bytes,
        'X_test': Xte_bytes,
        'y_test': yte_bytes,
    },
    'rtt': {
        'X_train': Xtr_rtt,
        'y_train': ytr_rtt,
        'X_test': Xte_rtt,
        'y_test': yte_rtt,
    },
}

results_default = []

for task_name, data in tasks.items():
    X_train = data['X_train']
    y_train = data['y_train']
    X_test = data['X_test']
    y_test = data['y_test']

    for name, model in models.items():
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])

        pipe.fit(X_train, y_train)
        results_default.append(
            regression_metrics(y_train, pipe.predict(X_train), f"{name}_{task_name}_train")
        )
        results_default.append(
            regression_metrics(y_test, pipe.predict(X_test), f"{name}_{task_name}_test")
        )

results_default = pd.DataFrame(results_default)
results_default


,model,MAE,RMSE,R2
0,LinearRegression_bytes_train,7359.401018,3.847812e+04,0.999908
1,LinearRegression_bytes_test,8027.511315,4.839652e+04,0.999973
2,RandomForest_bytes_train,2781.527457,2.701834e+05,0.995480
3,RandomForest_bytes_test,35987.627148,7.155753e+06,0.407872
4,HistGradientBoostingRegressor_bytes_train,43299.625393,1.020688e+06,0.935490
5,HistGradientBoostingRegressor_bytes_test,104476.138224,8.082028e+06,0.244654
6,LinearRegression_rtt_train,887.402494,2.906341e+03,0.094038
7,LinearRegression_rtt_test,1059.459626,3.116229e+03,0.035870
8,RandomForest_rtt_train,117.599531,6.501171e+02,0.954669
9,RandomForest_rtt_test,466.803529,2.155283e+03,0.538804


Cell 7 — Hyperparameter grids

In [8]:
param_grids = {
    'LinearRegression': {},

    'RandomForest': {
        # keep the search small to avoid getting stuck on the huge bytes task
        'model__n_estimators': [120, 200],
        'model__max_depth': [None, 20],
        'model__min_samples_leaf': [1, 4],
        'model__max_features': ['sqrt'],
        # subsample per tree to speed up training on 130k+ rows
        'model__max_samples': [0.4, 0.7]
    },

    'HistGradientBoostingRegressor': {
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [None, 8, 16],
        'model__max_iter': [200, 400]
    }
}


Cell 8 — Hyperparameter tuning (cross-validation)

In [10]:
results_tuned = []
best_params = {}

for task_name, data in tasks.items():
    X_train = data['X_train']
    y_train = data['y_train']
    X_test = data['X_test']
    y_test = data['y_test']

    for name, model in models.items():
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])

        
        cv_folds = 5
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grids[name],
            scoring='neg_mean_absolute_error',
            cv=cv_folds,
            n_jobs=-1,
            verbose=2
        )

        X_fit, y_fit = X_train, y_train
        # train RF on a subset to avoid extremely long runtimes on bytes
        if name == 'RandomForest' and len(y_train) > 70000:
            X_fit = X_train.sample(n=70000, random_state=42)
            y_fit = y_train.loc[X_fit.index]

        grid.fit(X_fit, y_fit)
        best_params[(task_name, name)] = grid.best_params_
        best = grid.best_estimator_

        results_tuned.append(
            regression_metrics(y_train, best.predict(X_train), f"{name}_{task_name}_train_tuned")
        )
        results_tuned.append(
            regression_metrics(y_test, best.predict(X_test), f"{name}_{task_name}_test_tuned")
        )

results_tuned = pd.DataFrame(results_tuned)
results_tuned


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits


,model,MAE,RMSE,R2
0,LinearRegression_bytes_train_tuned,7359.401018,3.847812e+04,0.999908
1,LinearRegression_bytes_test_tuned,8027.511315,4.839652e+04,0.999973
2,RandomForest_bytes_train_tuned,19904.313552,1.049131e+06,0.931844
3,RandomForest_bytes_test_tuned,66006.444092,7.982732e+06,0.263101
4,HistGradientBoostingRegressor_bytes_train_tuned,41098.058201,1.033395e+06,0.933874
5,HistGradientBoostingRegressor_bytes_test_tuned,101151.670962,8.090643e+06,0.243043
6,LinearRegression_rtt_train_tuned,887.402494,2.906341e+03,0.094038
7,LinearRegression_rtt_test_tuned,1059.459626,3.116229e+03,0.035870
8,RandomForest_rtt_train_tuned,337.773111,1.692813e+03,0.692649
9,RandomForest_rtt_test_tuned,618.600912,2.429108e+03,0.414171


Cell 9 — Final MAE comparison   

In [11]:

summary = results_tuned.copy()
summary['target'] = summary['model'].apply(lambda m: 's_bytes_all' if 'bytes' in m else 's_rtt_avg')
summary['split'] = summary['model'].apply(lambda m: 'train' if '_train' in m else 'test')

print('Best hyperparameters from CV:')
for (target, model), params in best_params.items():
    print(f"  {target} - {model}: {params}")

print('Tuned results:')
display(summary[['model', 'target', 'split', 'MAE', 'RMSE', 'R2']])

best_mae = summary[summary['split'] == 'test'].groupby('target')['MAE'].min()
print('Best MAE by target (test set):')
print(best_mae)
print('Units: MAE for s_bytes_all is in bytes; MAE for s_rtt_avg is in milliseconds.')


Best hyperparameters from CV:
  bytes - LinearRegression: {}
  bytes - RandomForest: {'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__max_samples': 0.7, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
  bytes - HistGradientBoostingRegressor: {'model__learning_rate': 0.05, 'model__max_depth': 8, 'model__max_iter': 200}
  rtt - LinearRegression: {}
  rtt - RandomForest: {'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__max_samples': 0.7, 'model__min_samples_leaf': 1, 'model__n_estimators': 120}
  rtt - HistGradientBoostingRegressor: {'model__learning_rate': 0.1, 'model__max_depth': None, 'model__max_iter': 200}
Tuned results:


,model,target,split,MAE,RMSE,R2
0,LinearRegression_bytes_train_tuned,s_bytes_all,train,7359.401018,3.847812e+04,0.999908
1,LinearRegression_bytes_test_tuned,s_bytes_all,test,8027.511315,4.839652e+04,0.999973
2,RandomForest_bytes_train_tuned,s_bytes_all,train,19904.313552,1.049131e+06,0.931844
3,RandomForest_bytes_test_tuned,s_bytes_all,test,66006.444092,7.982732e+06,0.263101
4,HistGradientBoostingRegressor_bytes_train_tuned,s_bytes_all,train,41098.058201,1.033395e+06,0.933874
5,HistGradientBoostingRegressor_bytes_test_tuned,s_bytes_all,test,101151.670962,8.090643e+06,0.243043
6,LinearRegression_rtt_train_tuned,s_rtt_avg,train,887.402494,2.906341e+03,0.094038
7,LinearRegression_rtt_test_tuned,s_rtt_avg,test,1059.459626,3.116229e+03,0.035870
8,RandomForest_rtt_train_tuned,s_rtt_avg,train,337.773111,1.692813e+03,0.692649
9,RandomForest_rtt_test_tuned,s_rtt_avg,test,618.600912,2.429108e+03,0.414171


Best MAE by target (test set):
target
s_bytes_all    8027.511315
s_rtt_avg       451.139337
Name: MAE, dtype: float64
Units: MAE for s_bytes_all is in bytes; MAE for s_rtt_avg is in milliseconds.
